# 01 — Data Exploration

Initial EDA for the F1 Analytics project. Covers:
- Loading data from FastF1, OpenF1, and Jolpica
- Data shapes, types, and missing values
- Sample telemetry plots
- Track map visualization

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import FastF1Loader, OpenF1Client, JolpicaClient
from src.visualization import TrackPlotter, TelemetryPlotter

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load a sample session via FastF1

In [ ]:
loader = FastF1Loader()

# Load the 2024 Monza race as our exploration session
session = loader.get_session(2024, 'Monza', 'R')
laps = session.laps

print(f'Session: {session.event["EventName"]} {session.name}')
print(f'Laps shape: {laps.shape}')
print(f'Columns: {list(laps.columns)}')

In [ ]:
# Overview of data types and missing values
print('--- Data Types ---')
print(laps.dtypes)
print('\n--- Missing Values ---')
print(laps.isnull().sum().sort_values(ascending=False).head(15))

## 2. Lap time distribution

In [ ]:
# Filter out pit laps and outliers
from src.analysis import LapAnalyzer

analyzer = LapAnalyzer()
clean_laps = analyzer.filter_representative_laps(laps)

print(f'Total laps: {len(laps)}, Representative laps: {len(clean_laps)}')

# Lap time distribution per driver
fig = analyzer.plot_lap_distributions(clean_laps, kind='violin', top_n=10)
plt.show()

## 3. Sample telemetry

In [ ]:
# Get fastest lap telemetry for the race winner
fastest = laps.pick_fastest()
tel = fastest.get_telemetry()

print(f'Fastest lap by: {fastest["Driver"]}')
print(f'Telemetry shape: {tel.shape}')
print(f'Telemetry columns: {list(tel.columns)}')
tel.head()

In [ ]:
# Multi-channel telemetry plot
plotter = TelemetryPlotter()
fig = plotter.plot_telemetry_channels(tel, title=f'Fastest Lap — {fastest["Driver"]}')
plt.show()

## 4. Track map

In [ ]:
# Speed-colored track map
fig = TrackPlotter.plot_speed_map(tel, title=f'Monza Speed Map — {fastest["Driver"]}')
plt.show()

In [ ]:
# Gear map
fig = TrackPlotter.plot_gear_map(tel, title=f'Monza Gear Map — {fastest["Driver"]}')
plt.show()

## 5. Compare two drivers

In [ ]:
# Compare VER and NOR fastest laps
drivers = ['VER', 'NOR']
tel_dict = {}
for drv in drivers:
    try:
        fl = laps.pick_driver(drv).pick_fastest()
        tel_dict[drv] = fl.get_telemetry()
    except Exception as e:
        print(f'Could not load {drv}: {e}')

fig = plotter.plot_multi_channel_comparison(tel_dict)
plt.show()

## 6. Historical data from Jolpica

In [ ]:
jolpica = JolpicaClient()

# Get 2024 race results
results_2024 = jolpica.get_race_results(2024)
print(f'2024 results shape: {results_2024.shape}')
results_2024.head(10)

In [ ]:
# Driver standings
standings = jolpica.get_driver_standings(2024)
print(standings.to_string(index=False))

## 7. OpenF1 sample

In [ ]:
openf1 = OpenF1Client()

# Get 2024 sessions list
sessions = openf1.get_sessions(year=2024)
print(f'Sessions shape: {sessions.shape}')
sessions.head(10)